In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


# 1. SETUP: Standard Cleaning Function
# This fixes the "dirty columns" you saw in SQL (trimming spaces, handling empty strings)
def clean_text(df):
    """Trims whitespace and converts empty strings to NULL for all String columns"""
    for col_name in df.columns:
        if isinstance(df.schema[col_name].dataType, StringType):
            df = df.withColumn(col_name, trim(col(col_name))) \
                   .withColumn(col_name, when(col(col_name) == "", None).otherwise(col(col_name)))
    return df

print("🚀 Cleaning Engine Loaded.")

StatementMeta(, b67e0544-e03f-4ebb-823d-0df98fe61745, 3, Finished, Available, Finished, False)

🚀 Cleaning Engine Loaded.


In [2]:
from pyspark.sql.functions import col, trim, lower, current_timestamp, first

# --- STEP 1: Define Helper Functions ---
# This fixes the "clean_text is not defined" error
def clean_text(df):
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, trim(lower(col(col_name))))
    return df

# --- STEP 2: CUSTOMERS (The Foundation) ---
print("🔨 Cleaning Customers...")
df_cust = spark.read.table("bronze_customers")
df_cust = clean_text(df_cust) 

# IMPORTANT: We keep customer_id so the Fact Table can join to it later!
df_silver_customers = df_cust.select(
    "customer_id", 
    "customer_unique_id", 
    col("customer_city").alias("city"), 
    col("customer_state").alias("state"), 
    col("customer_zip_code_prefix").alias("zip_code")
).withColumn("ingest_time", current_timestamp())

df_silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_customers")

print(f"✅ silver_customers: {df_silver_customers.count()} rows")

# --- STEP 3: PRODUCTS (Handling Nulls) ---
print("🔨 Cleaning Products...")
df_prod = spark.read.table("bronze_products")
df_prod = clean_text(df_prod)

# Fill Missing Values (Fixes the NULLs you identified earlier)
df_silver_products = df_prod \
    .na.fill("Unknown", subset=["product_category_name"]) \
    .na.fill(0, subset=["product_weight_g", "product_length_cm"]) \
    .withColumnRenamed("product_id", "prod_id") 

df_silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_products")

print(f"✅ silver_products: {df_silver_products.count()} rows")

StatementMeta(, e7f06d9f-6e34-4ff7-afc0-a4474d37a14d, 4, Finished, Available, Finished, False)

🔨 Cleaning Customers...
✅ silver_customers: 99441 rows
🔨 Cleaning Products...
✅ silver_products: 32951 rows


In [4]:
from pyspark.sql.functions import col, trim, lower, current_timestamp, first, to_timestamp

# --- ORDERS: Date Casting ---
print("🔨 Cleaning Orders...")
df_ord = spark.read.table("bronze_orders")
df_ord = clean_text(df_ord)

# Convert Strings to Timestamps for Time Intelligence
df_silver_orders = df_ord \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date")) \
    .withColumn("ingest_time", current_timestamp())

df_silver_orders.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_orders")
print(f"✅ silver_orders: {df_silver_orders.count()} rows")

# --- ORDER ITEMS: Fixing Dirty IDs ---
print("🔨 Cleaning Order Items...")
df_items = spark.read.table("bronze_order_items")
df_items = clean_text(df_items) 

# Explicit Cast to ensure they match the primary keys
df_silver_items = df_items \
    .withColumn("order_id", col("order_id").cast("string")) \
    .withColumn("product_id", col("product_id").cast("string")) \
    .withColumn("seller_id", col("seller_id").cast("string")) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

df_silver_items.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_order_items")
print(f"✅ silver_order_items: {df_silver_items.count()} rows")

StatementMeta(, e7f06d9f-6e34-4ff7-afc0-a4474d37a14d, 6, Finished, Available, Finished, False)

🔨 Cleaning Orders...
✅ silver_orders: 99441 rows
🔨 Cleaning Order Items...
✅ silver_order_items: 112650 rows


In [5]:
# VERIFICATION: Bronze vs Silver Row Counts
tables = [
    ("orders", "bronze_orders", "silver_orders"),
    ("items", "bronze_order_items", "silver_order_items")
]

print(f"{'TABLE':<10} | {'BRONZE':<10} | {'SILVER':<10} | {'STATUS'}")
print("-" * 50)

for name, bronze_tbl, silver_tbl in tables:
    b_count = spark.read.table(bronze_tbl).count()
    s_count = spark.read.table(silver_tbl).count()
    
    status = "✅ Match" if b_count == s_count else f"⚠️ Diff: {b_count - s_count}"
    print(f"{name:<10} | {b_count:<10} | {s_count:<10} | {status}")

StatementMeta(, e7f06d9f-6e34-4ff7-afc0-a4474d37a14d, 7, Finished, Available, Finished, False)

TABLE      | BRONZE     | SILVER     | STATUS
--------------------------------------------------
orders     | 99441      | 99441      | ✅ Match
items      | 112650     | 112650     | ✅ Match


In [6]:
from pyspark.sql.functions import avg, first, regexp_replace

print("🔨 Processing Supporting Tables...")

# 1. SELLERS (Clean & Save)
df_sellers = spark.read.table("bronze_sellers")
df_sellers = clean_text(df_sellers)
df_sellers.write.format("delta").mode("overwrite").saveAsTable("silver_sellers")
print(f"✅ silver_sellers created: {df_sellers.count()} rows")

# 2. PAYMENTS (Clean & Save)
df_pay = spark.read.table("bronze_order_payments")
df_pay = clean_text(df_pay)
df_pay.write.format("delta").mode("overwrite").saveAsTable("silver_order_payments")
print(f"✅ silver_order_payments created: {df_pay.count()} rows")

# 3. REVIEWS (Remove newlines in comments to prevent breaking downstream tools)
df_rev = spark.read.table("bronze_order_reviews")
df_rev = clean_text(df_rev)
df_rev = df_rev.withColumn("review_comment_message", regexp_replace("review_comment_message", "\n", " "))
df_rev.write.format("delta").mode("overwrite").saveAsTable("silver_order_reviews")
print(f"✅ silver_order_reviews created: {df_rev.count()} rows")

# 4. GEOLOCATION (Deduplicate Zip Codes)
# Logic: We group by Zip and take the 'Centroid' (Average Lat/Long)
df_geo = spark.read.table("bronze_geolocation")
df_silver_geo = df_geo.groupBy("geolocation_zip_code_prefix") \
    .agg(
        avg("geolocation_lat").alias("lat"),
        avg("geolocation_lng").alias("lng"),
        first("geolocation_city").alias("city"),
        first("geolocation_state").alias("state")
    )
df_silver_geo.write.format("delta").mode("overwrite").saveAsTable("silver_geolocation")
print(f"✅ silver_geolocation created: {df_silver_geo.count()} unique zips")

StatementMeta(, e7f06d9f-6e34-4ff7-afc0-a4474d37a14d, 8, Finished, Available, Finished, False)

🔨 Processing Supporting Tables...
✅ silver_sellers created: 3095 rows
✅ silver_order_payments created: 103886 rows
✅ silver_order_reviews created: 104162 rows
✅ silver_geolocation created: 19015 unique zips
